In [1]:
import coiled

import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import pytz
import dask
import re
import sparse
import time
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy

import pygwalker as pyg

# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

In [2]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [3]:
fs = fsspec.filesystem("s3", requester_pays=True)

In [ ]:
cluster = coiled.Cluster(
    name="LULUCF_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=20,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r5.2xlarge", # memory optimized AWS EC2 instances
    worker_vm_types="r5.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

In [ ]:
client.restart() 

In [4]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 7
Total threads: 14,Total memory: 31.08 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44419,Workers: 7
Dashboard: http://127.0.0.1:8787/status,Total threads: 14
Started: Just now,Total memory: 31.08 GiB
Comm: tcp://127.0.0.1:39007,Total threads: 2
Dashboard: http://127.0.0.1:45861/status,Memory: 4.44 GiB
Nanny: tcp://127.0.0.1:41197,


In [52]:
local_client.shutdown()

2025-05-14 12:33:02,508 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/home/dagibbs22/miniforge3/envs/coiled_20250514/lib/python3.10/site-packages/distributed/comm/tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/dagibbs22/miniforge3/envs/coiled_20250514/lib/python3.10/site-packages/distributed/worker.py", line 1269, in heartbeat
    response = await retry_operation(
  File "/home/dagibbs22/miniforge3/envs/coiled_20250514/lib/python3.10/site-packages/distributed/utils_comm.py", line 416, in retry_operation
    return await retry(
  File "/home/dagibbs22/miniforge3/envs/coiled_20250514/lib/python3.10/site-packages/distributed/utils_comm.py", line 395, in retry
    return await coro()
  File "

In [5]:
def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [6]:
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

In [7]:
# Node codes output from model. Covers entire decision tree. Make sure that node codes are right-padded with 0s to 7 digits! 
# Otherwise, only the node codes that are seven digits without 0s will be matched with the node code rasters and output. 
# TODO: I may have accidentally missed some node codes when copying them from the decision tree. Check!
node_codes = np.array([
    1110000, 1120000, 1210000, 1220000, 2111000, 2112000,
    2121100, 2121200, 2122100, 2122200, 2123100, 2123200,
    2124100, 2124200, 2125100, 2125200, 2211100, 2211200, 2212110, 2212120, 
    2212210, 2212220, 2213110, 2213120, 2213210, 2213220,
    2214100, 2214200, 2215100, 2215200, 2221100, 2221200, 2223100, 2223200,
    2222100, 2222200, 3110000, 3120000, 3211211, 3211212,
    3211221, 3211222, 3212111, 3212112, 3212121, 3212122,
    3212211, 3212212, 3212221, 3212222, 3221110, 3221120,
    3221210, 3221220, 3222111, 3222112, 3222121, 3222122,
    3222210, 3222220, 4100000, 4210000, 4220000, 4310000,
    4320000, 5100000, 5210000, 5220000, 5310000, 5320000],
dtype=np.uint32)

# # Node codes output from model for 2x2 test area in DRC (23_-5_25_-3)
# node_codes = np.array([3222111, 2212120, 3222121, 3212122, 2212110, 3212121, 3222210, 2223200, 
#                        2221200, 3212222, 2221100, 5220000, 4100000, 2223100,
#                        2212220, 3212221, 2211200, 5210000, 2211100, 4220000, 2212210, 2214100, 2214200, 4210000, 2215200], 
#                       dtype=np.uint32)

In [8]:
# Extracts some metadata/chunk properties to add to the output dataframe
def parse_metadata_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__(\d+_-?\d+_\d+_-?\d+)__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_pixel_yr_(\d{4}_\d{4})\.tif$"
    match = re.search(pattern, uri)

    if match:
        chunk_id = match.group(1)
        variable = match.group(2)
        interval = match.group(3)
    else:
        interval, chunk_id, variable = None, None, None

    return interval, chunk_id, variable

In [9]:
def make_xarray_chunks(tile_uris):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True,
        chunks={'x': 4000, 'y':4000}
    ).squeeze().persist()

    return xarray_chunks

In [10]:
def align_with_nodes(analysis_layer, nodes):
    analysis_layer_sub, nodes_aligned = xr.align(analysis_layer, nodes, join="inner")
    return analysis_layer_sub, nodes_aligned

In [11]:
def xarray_reduction_sum_count(analysis_layer, node_data):

    reductions = {}

    for func in ["sum", "count"]:
        reduced = xarray_reduce(
            analysis_layer.band_data,
            node_data,
            func=func,
            keep_attrs=True,
            expected_groups=(node_codes),
            reindex=ReindexStrategy(
                blockwise=False,
                array_type=ReindexArrayType.SPARSE_COO
            ),
            fill_value=0
        )

        # Rename variables to reflect reduction type
        if isinstance(reduced, xr.Dataset):
            renamed = reduced.rename({var: f"{var}_{func}" for var in reduced.data_vars})
        else:  # it's a DataArray
            renamed = reduced.rename(f"{reduced.name}_{func}")

        reductions[func] = renamed

    # Merge results: handle Dataset or DataArray combinations
    result = xr.merge([r if isinstance(r, xr.Dataset) else r.to_dataset() for r in reductions.values()])

    return result

In [12]:
def xarray_reduction(analysis_layer, node_data):

    analysis_layer_by_node = xarray_reduce(
        analysis_layer,
        node_data,
        func='sum',
        keep_attrs=True,
        expected_groups=(node_codes),
        reindex=ReindexStrategy(
            blockwise=False, array_type=ReindexArrayType.SPARSE_COO
        ),
        fill_value=0   
    )

    return analysis_layer_by_node

In [13]:
def create_output_year_df(output_year_result, interval, ouput_pattern):

    output_year_result_sparse_data = output_year_result.data

    # Step 3: Extract coordinates and values
    dim_names = output_year_result.dims
    indices = output_year_result_sparse_data.coords
    output_year_values = output_year_result_sparse_data.data

    # Step 4: Map dimension indices to coordinate values
    output_year_coord_dict = {
        dim: output_year_result.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    output_year_coord_dict["value"] = output_year_values
    
    output_year_coord_dict = {
        dim: output_year_result.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    output_year_coord_dict["value"] = output_year_values
    
    output_year_df = pd.DataFrame(output_year_coord_dict)

    output_year_df["interval_end"] = interval
    output_year_df["output_pattern"] = ouput_pattern

    return output_year_df

In [14]:
# Reclassifies state nodes to broad categories
def classify_node(state_node):
    
    node_str = str(state_node)
    first_digit = int(node_str[0])
    # print(first_digit)

    # For broad classes that can be categorized using just the first digit
    one_digit_map = {
        1: 'forest_gain',
        2: 'forest_loss',
        4: 'cropland',
        5: 'grassland'
    }

    # For broad classes that need to be categorized using the first three digits
    three_digit_map = {
        321: 'disturbed_forest',
        322: 'stable_forest'
        # Add more as needed
    }
    
    if first_digit == 3:
        prefix = int(node_str[:3])
        # print(prefix)
        # print(two_digit_map.get(prefix, 'unknown_3x'))
        return three_digit_map.get(prefix, 'unknown_3x')
    else:
        return one_digit_map.get(first_digit, 'unknown')

Code to run zonal stats

In [32]:
# uri components

model_version = "version_0_3_2"
run_date = "20250507"
# model_version = "version_0_3_3"
# run_date = "20250511"
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"
interval_end_years = [2016]
# interval_end_years = [2016, 2017, 2018]
# interval_end_years = [2020]
# interval_end_years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

# s3 folders for inputs
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/"

node_folder = f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/4000_pixels/{run_date}/"

# analysis_layers_folders = [gross_emis_CO2_folder, gross_emis_all_gases_folder, gross_remv_all_pools_folder, net_flux_all_pools_CO2_folder]
analysis_layers_folders = [net_flux_all_pools_CO2_folder]


In [47]:
combined_df = pd.DataFrame()
analysis_start_time = time.time()

interval_end_year = 2020
output_pattern = 'gross_emissions__all_C_pools__CO2_only'
focal_analysis_layer_folder = gross_emis_CO2_folder

interval = f"{interval_end_year-1}_{interval_end_year}"
# print(interval)

# Creates a Pandas series of s3 uris for this specific analysis layer
gross_emis_CO2_folder = gross_emis_CO2_folder.replace("INTERVAL", interval)
gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder)
# print(gross_emis_CO2_folder)
# print(gross_emis_CO2_uris[0])

gross_remv_all_pools_folder = gross_emis_CO2_folder.replace("INTERVAL", interval)
gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder)
# print(gross_remv_all_pools_folder)
# print(gross_remv_all_pools_uris[0])

net_flux_all_pools_CO2_folder = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval)
net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder)
# print(net_flux_all_pools_CO2_folder)
# print(net_flux_all_pools_CO2_uris[0])

# Creates a Pandas series of s3 uris for the relevant node codes
node_folder = node_folder.replace("INTERVAL", interval)
node_tile_year_uris = list_folder_uris(node_folder)
# print(node_folder)
# print(node_tile_year_uris[0])

# Gets input layer metadata, like the output pattern.
# Note: chunk_id is for the first chunk being processed, not all chunks being processed.
gross_emis_CO2_interval_from_inputs, gross_emis_CO2_chunk_id, gross_emis_CO2_output_pattern = parse_metadata_from_uri(gross_emis_CO2_uris)
net_flux_interval_from_inputs, net_flux_chunk_id, net_flux_output_pattern = parse_metadata_from_uri(net_flux_all_pools_CO2_uris)
gross_remv_all_pools_interval_from_inputs, gross_remv_all_pools_chunk_id, gross_remv_all_pools_output_pattern = parse_metadata_from_uri(gross_remv_all_pools_uris)
# print(gross_emis_CO2_output_pattern)
# print(net_flux_output_pattern)

print(f"Processing {interval} from {focal_analysis_layer_folder}: {timestr()}")
layer_interval_start_time = time.time()

print(f"   Reading {interval}: {timestr()}")
gross_emis_CO2_xarray_chunks = make_xarray_chunks(gross_emis_CO2_uris)
gross_remv_all_pools_xarray_chunks = make_xarray_chunks(gross_remv_all_pools_uris)
net_flux_all_pools_CO2_xarray_chunks = make_xarray_chunks(net_flux_all_pools_CO2_uris)
node_xarray_chunks = make_xarray_chunks(node_tile_year_uris)
# print("layer_xarray_chunks:", layer_xarray_chunks)
# print("nodes_xarray_chunks:", nodes_xarray_chunks)

print(f"   Aligning {interval}: {timestr()}")
gross_emis_CO2_aligned, nodes_aligned = align_with_nodes(gross_emis_CO2_xarray_chunks, node_xarray_chunks)
gross_remv_all_pools_aligned, nodes_aligned = align_with_nodes(gross_remv_all_pools_xarray_chunks, node_xarray_chunks)
net_flux_all_pools_CO2_aligned, nodes_aligned = align_with_nodes(net_flux_all_pools_CO2_xarray_chunks, node_xarray_chunks)
# print("gross_emis_CO2_aligned:", gross_emis_CO2_aligned)
# print("gross_remv_all_pools_aligned:", gross_remv_all_pools_aligned)
# print("net_flux_all_pools_CO2_aligned:", net_flux_all_pools_CO2_aligned)
# print("nodes_aligned:", nodes_aligned)

nodes_aligned_data = nodes_aligned.band_data
nodes_aligned_data.name = 'state_node'

flux_cube = xr.DataArray(dask.array.stack((gross_emis_CO2_aligned.band_data, gross_remv_all_pools_aligned.band_data, net_flux_all_pools_CO2_aligned.band_data)), dims=('flux_type', 'y', 'x'))
# flux_cube

Processing 2019_2020 from s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_2/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/2019_2020/_pixel_yr/4000_pixels/20250507/: 20250514_12_27_06
   Reading 2019_2020: 20250514_12_27_06
   Aligning 2019_2020: 20250514_12_27_08


In [48]:
data_cube_by_node = xarray_reduce(
    flux_cube,
    nodes_aligned_data,
    func='sum',
    keep_attrs=True,
    expected_groups=(node_codes),
    reindex=ReindexStrategy(
        blockwise=False, array_type=ReindexArrayType.SPARSE_COO
    ),
    fill_value=0   
)

data_cube_by_node

<xarray.DataArray 'stack-cbe449f063316b6007c6ac502e16fb01' (flux_type: 3,
                                                            state_node: 70)> Size: 840B
dask.array<groupby_nansum, shape=(3, 70), dtype=float32, chunksize=(1, 70), chunktype=sparse.COO>
Coordinates:
    band         int64 8B 1
    spatial_ref  int64 8B 0
  * state_node   (state_node) uint32 280B 1110000 1120000 ... 5310000 5320000
Dimensions without coordinates: flux_type

In [49]:
result = data_cube_by_node.compute()

In [51]:
sparse_data = result.data

# Step 3: Extract coordinates and values
dim_names = result.dims
indices = sparse_data.coords
values = sparse_data.data

# Step 4: Map dimension indices to coordinate values
coord_dict = {
    dim: result.coords[dim].values[indices[i]]
    for i, dim in enumerate(dim_names)
}
coord_dict["value"] = values

df = pd.DataFrame(coord_dict)
df['flux_type'] = df['flux_type'].replace({0: 'gross_emissions__all_C_pools__CO2_only', 1: 'gross_removals__all_C_pools', 2: 'net_flux__all_C_pools__CO2_only'})
df['node_grp'] = df['state_node'].apply(classify_node)
df['state_node'] = 'n' + df['state_node'].astype(str)
df

,flux_type,state_node,value,node_grp
0,gross_emissions__all_C_pools__CO2_only,n2211100,1.217371e+04,forest_loss
1,gross_emissions__all_C_pools__CO2_only,n2211200,2.277452e+04,forest_loss
2,gross_emissions__all_C_pools__CO2_only,n2212110,8.374514e+06,forest_loss
3,gross_emissions__all_C_pools__CO2_only,n2212120,3.175806e+07,forest_loss
4,gross_emissions__all_C_pools__CO2_only,n2212210,1.725815e+04,forest_loss
...,...,...,...,...
76,net_flux__all_C_pools__CO2_only,n4210000,1.441859e+03,cropland
77,net_flux__all_C_pools__CO2_only,n4220000,2.536077e+03,cropland
78,net_flux__all_C_pools__CO2_only,n5100000,-5.714232e+03,grassland
79,net_flux__all_C_pools__CO2_only,n5210000,1.340556e+04,grassland
